In [9]:
%pip install pandas
import pandas as pd


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 22.0.4 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# 1. Load the summary and stats files
summary_df = pd.read_csv(
    'ggood3_project_summary.csv',
    sep=';',
    header=None,
    names=['project', 'sha1', 'author', 'timestamp', 'message'],
    engine='python',
)
stats_df = pd.read_csv('ggood3_project_stats.csv', sep=';')

# Convert timestamps to datetime for accurate filtering around gaps
summary_df['datetime'] = pd.to_datetime(summary_df['timestamp'], unit='s')


# 2. Define commit message theme classifier
def classify_theme(message):
  if not isinstance(message, str):
    return 'Other'
  msg = message.lower()

  if any(
      b in msg
      for b in [
          'dependabot',
          'bot',
          '[bot]',
          'github-actions',
          'pre-commit.ci',
          'auto-format',
      ]
  ):
    return 'Automated bot contributions'
  if any(
      s in msg for s in ['security', 'cve-', 'vulnerability', 'sanitize', 'patch']
  ):
    return 'Security patches'
  if any(
      d in msg
      for d in [
          'dependency',
          'dependencies',
          'upgrade',
          'bump',
          'npm',
          'pip',
          'requirements',
          'cargo.lock',
          'yarn.lock',
      ]
  ):
    return 'Dependency updates'
  if any(
      r in msg
      for r in [
          'release',
          'version',
          'bump version',
          'v1.',
          'v2.',
          'changelog',
          'tag',
      ]
  ):
    return 'Release'
  if any(
      f in msg
      for f in [
          'fix',
          'bug',
          'issue',
          'error',
          'patch',
          'crash',
          'fail',
          'resolve',
          'correct',
          'fault',
          'hotfix',
      ]
  ):
    return 'Bug fixes'
  if any(
      doc in msg
      for doc in [
          'doc',
          'readme',
          'license',
          'comment',
          'typo',
          'contributing',
          'wiki',
      ]
  ):
    return 'Documentation updates'
  if any(
      feat in msg
      for feat in [
          'feat',
          'feature',
          'add',
          'implement',
          'support',
          'new',
          'create',
          'introduce',
          'extend',
          'enable',
      ]
  ):
    return 'Feature development'
  return 'Other'


summary_df['theme'] = summary_df['message'].apply(classify_theme)

gap_commits_list = []
before_themes_list = []
after_themes_list = []

# 3. Process each project
for idx, row in stats_df.iterrows():
  proj = row['project']
  gap_start = row['LongestGapStart (YYYY-MM)']
  gap_end = row['LongestGapEnd (YYYY-MM)']

  proj_commits = (
      summary_df[summary_df['project'] == proj]
      .sort_values('timestamp')
      .reset_index(drop=True)
  )

  # Collect up to 10 commits before the longest gap start
  gap_start_dt = pd.to_datetime(gap_start + '-01')
  pre_commits = proj_commits[proj_commits['datetime'] < gap_start_dt]
  last_10_pre = pre_commits.tail(10).copy()
  last_10_pre.insert(0, 'pre_post', 'pre')

  # Collect up to 10 commits after the longest gap end
  gap_end_dt = pd.to_datetime(gap_end + '-01')
  post_commits = proj_commits[proj_commits['datetime'] >= gap_end_dt]
  first_10_post = post_commits.head(10).copy()
  first_10_post.insert(0, 'pre_post', 'post')

  gap_commits_list.append(last_10_pre)
  gap_commits_list.append(first_10_post)

  # Determine top 2 themes before the gap
  if len(last_10_pre) > 0:
    top_pre = last_10_pre['theme'].value_counts().head(2).index.tolist()
    while len(top_pre) < 2:
      top_pre.append('Other')
  else:
    top_pre = ['Other', 'Other']
  before_themes_list.append(f'{top_pre[0]}:{top_pre[1]}')

  # Determine top 2 themes after the gap
  if len(first_10_post) > 0:
    top_post = first_10_post['theme'].value_counts().head(2).index.tolist()
    while len(top_post) < 2:
      top_post.append('Other')
  else:
    top_post = ['Other', 'Other']
  after_themes_list.append(f'{top_post[0]}:{top_post[1]}')

# 4. Save gap commits to ggood3_project_gap_commits.csv
all_gap_commits = pd.concat(gap_commits_list, ignore_index=True)
out_cols = ['pre_post', 'project', 'sha1', 'author', 'timestamp', 'message']
all_gap_commits[out_cols].to_csv(
    'ggood3_project_gap_commits.csv', sep=';', header=False, index=False, quotechar='"'
)

# 5. Update stats file with BeforeThemes and AfterThemes
stats_df['BeforeThemes'] = before_themes_list
stats_df['AfterThemes'] = after_themes_list
stats_df.to_csv('ggood3_project_stats.csv', sep=';', index=False)

print('Part 3.1 analysis complete. Files generated/updated successfully!')

Part 3.1 analysis complete. Files generated/updated successfully!
